In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
matches = pd.read_csv("../data/raw/international_results.csv")

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)

matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
def get_k_factor(tournament):
    if tournament == "FIFA World Cup":
        return 40

    if "qualification" in tournament.lower():
        return 30

    if tournament == "Friendly":
        return 10

    return 20

elo = {}

def get_rating(team):
    if team not in elo:
        elo[team] = 1500

    return elo[team]


def expected_score(team_rating, opponent_rating):
    return 1 / (1 + 10 ** ((opponent_rating - team_rating) / 400))


def actual_score(team_goals, opponent_goals):
    if team_goals > opponent_goals:
        return 1

    if team_goals == opponent_goals:
        return 0.5

    return 0


def update_rating(old_rating, expected, actual, k):
    return old_rating + k * (actual - expected)




In [6]:
home_elo_before = []
away_elo_before = []

for _, match in matches.iterrows():

    home_team = match["home_team"]
    away_team = match["away_team"]

    home_rating = get_rating(home_team)
    away_rating = get_rating(away_team)

    home_elo_before.append(home_rating)
    away_elo_before.append(away_rating)

    expected_home = expected_score(home_rating, away_rating)
    expected_away = expected_score(away_rating, home_rating)

    actual_home = actual_score(
        match["home_score"],
        match["away_score"]
    )

    actual_away = actual_score(
        match["away_score"],
        match["home_score"]
    )

    k = get_k_factor(match["tournament"])

    elo[home_team] = update_rating(
        home_rating,
        expected_home,
        actual_home,
        k
    )

    elo[away_team] = update_rating(
        away_rating,
        expected_away,
        actual_away,
        k
    )

matches["home_elo_before"] = home_elo_before
matches["away_elo_before"] = away_elo_before

matches["elo_difference"] = (
    matches["home_elo_before"]
    - matches["away_elo_before"]
)
matches[
    [
        "home_team",
        "away_team",
        "tournament",
        "home_elo_before",
        "away_elo_before",
        "elo_difference",
    ]
].head(10)

,home_team,away_team,tournament,home_elo_before,away_elo_before,elo_difference
0,Scotland,England,Friendly,1715.126652,1894.773688,-179.647036
1,England,Scotland,Friendly,1892.396527,1717.503812,174.892714
2,Scotland,England,Friendly,1714.827675,1895.072664,-180.244989
3,England,Scotland,Friendly,1887.688849,1722.211491,165.477358
4,Scotland,England,Friendly,1724.427796,1885.472543,-161.044747
5,Scotland,Wales,Friendly,1731.592557,1687.292524,44.300033
6,England,Scotland,Friendly,1878.307783,1735.958460,142.349322
7,Wales,Scotland,Friendly,1682.926620,1742.899611,-59.972991
8,Scotland,England,Friendly,1747.045001,1871.366632,-124.321631
9,Scotland,Wales,Friendly,1753.761491,1678.781230,74.980262


In [7]:
matches["home_advantage"] = (~matches["neutral"]).astype(int)
def match_result(home_score, away_score):
    if home_score > away_score:
        return "home_win"

    if home_score < away_score:
        return "away_win"

    return "draw"


matches["result"] = matches.apply(
    lambda row: match_result(row["home_score"], row["away_score"]),
    axis=1
)

model_data = matches[
    [
        "date",
        "home_team",
        "away_team",
        "tournament",
        "elo_difference",
        "home_advantage",
        "result",
    ]
].copy()
train = model_data[model_data["date"] < "2018-01-01"]
test = model_data[model_data["date"] >= "2018-01-01"]

features = [
    "elo_difference",
    "home_advantage",
]

X_train = train[features]
y_train = train["result"]

X_test = test[features]
y_test = test["result"]

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)
accuracy = accuracy_score(y_test, predictions)
loss = log_loss(y_test, probabilities, labels=model.classes_)

print("Accuracy:", accuracy)
print("Log Loss:", loss)

Accuracy: 0.5990974509086474
Log Loss: 0.8736935137427563


In [9]:
baseline_prediction = y_train.value_counts().idxmax()

baseline_predictions = [baseline_prediction] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Baseline prediction:", baseline_prediction)
print("Baseline accuracy:", baseline_accuracy)
print("Model accuracy:", accuracy)
matches.to_csv(
    "../data/processed/matches_with_improved_elo.csv",
    index=False
)

Baseline prediction: home_win
Baseline accuracy: 0.4768874252957678
Model accuracy: 0.5990974509086474


In [10]:
elo_table = pd.DataFrame(
    sorted(elo.items(), key=lambda item: item[1], reverse=True),
    columns=["team", "elo_rating"]
)

elo_table.to_csv(
    "../data/processed/latest_elo_ratings.csv",
    index=False
)

In [11]:
import joblib

joblib.dump(
    model,
    "../models/improved_logistic_regression_model.pkl"
)

['../models/improved_logistic_regression_model.pkl']